In [ ]:
import sys
sys.path.append("/Users/ronguy/Dropbox/Work/CyTOF/")
sys.path.append("/Users/ronguy/Dropbox/Work/CyTOF/Code/")
%load_ext autoreload
%autoreload 2
from CyTOFHelper import *
from TestHet import *
from PermCell_Smooth import *
#from SHAPset import *
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
Run="Corrs"
import xgboost as xgb
import umap
import matplotlib.pyplot as plt
import anndata as ad
import scanpy as sc
import pandas as pd
from POgSET import *
import icecream as ic

In [ ]:
%matplotlib inline

In [ ]:
import glob

In [ ]:
type_="cytoff"
dir="/Users/ronguy/Dropbox/CyTOF_Breast/data/guy/with_labels/cytoff/"

In [ ]:
Numbs=[5,6,7,8]

In [ ]:
Numbs.sort()
Numbs

In [ ]:
DBs=[f"PDX{N}" for N in Numbs]
DBs

In [ ]:
for DB,N in zip(DBs,Numbs):
    print(DB)
    globals()[DB]=pd.read_parquet(f"/Users/ronguy/Dropbox/CyTOF_Breast/data/guy/with_labels/pdx/normalized_not_scaled_{N}.0.parquet")
    print(DB,globals()[DB].shape)

In [ ]:
#dir="/Users/ronguy/Dropbox/WIS-CIMA colab - Analysis/#3 CyTOF  - KPC sample, after CD45 depletion/"

params = {'axes.titlesize': 30,
          'legend.fontsize': 20,
          'figure.figsize': (6, 5),
          'axes.labelsize': 20,
          'axes.titlesize': 20,
          'xtick.labelsize': 20,
          'ytick.labelsize': 20,
          'figure.titlesize': 30}
plt.rcParams.update(params)

sns.set_style("white")


In [ ]:
Rep=dict(pd.read_excel("/Users/ronguy/Dropbox/Work/CyTOF/Mapping.xlsx").iloc[:,:].values)

In [ ]:
Rep

In [ ]:
for DB in DBs:
#    globals()[DB]=pd.read_csv(F)
    globals()[DB].rename(columns=Rep,inplace=True)
    try:
        globals()[DB].drop(columns=['DNA1','DNA2','Event #'],inplace=True)        
    except:
        pass




In [ ]:
N=list(globals()[DBs[0]].columns)
N.sort()
#N.remove('N-cadherin')
NamesAll,EpiCols,NormMRK,CellIden,CellCyle=GetMarkers(N)
NamesAll.remove("class")

In [ ]:
NamesAll

In [ ]:
for DB in DBs:
    globals()[f"Label_{DB}"]=globals()[DB]["class"].values

In [ ]:
for DB in DBs:
    globals()[DB]=globals()[DB][NamesAll]
#    globals()[DB]['Samp']=DB

In [ ]:
NC=1300
aaaa=pd.DataFrame(columns=globals()[DBs[0]].columns)
for DB in DBs:
    aaaa=pd.concat([aaaa,globals()[DB].sample(NC,replace=False)]).copy()
                  

                
m=np.mean(aaaa,axis=0)
s=np.std(aaaa,axis=0)

for DB in DBs:
    # m=globals()[DB].mean()
    # s=globals()[DB].std()
    globals()[DB]=(globals()[DB]-m)/s
    globals()[DB]['Samp']=DB
    globals()[DB]['Class']=globals()[f"Label_{DB}"]

In [ ]:
%matplotlib inline
hKWD={'element':'step','fill':False,'stat':'density'}

In [ ]:
for DB in DBs:

    globals()[DB]['Samp']=DB
    print(DB,globals()[DB].shape[0])

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity, rbf_kernel
from sklearn.preprocessing import StandardScaler
from vendi_score import vendi
def _kernel_matrix(X, kernel="cosine", sigma=None, subsample_for_sigma=5000, random_state=0):
    if kernel == "cosine":
        return cosine_similarity(X)  # diag = 1
    elif kernel == "rbf":
        if sigma is None:
            rng = np.random.default_rng(random_state)
            idx = rng.choice(X.shape[0], min(subsample_for_sigma, X.shape[0]), replace=False)
            D = np.square(X[idx][:, None, :] - X[None, idx, :]).sum(-1)
            # median of pairwise Euclidean distances
            med = np.sqrt(np.median(D[D > 0]))
            sigma = med/3. if med > 0 else 1.0
        gamma = 1.0 / (2.0 * sigma**2)
        return rbf_kernel(X, gamma=gamma)
    else:
        raise ValueError("kernel must be 'cosine' or 'rbf'")

def vendi_score(X, kernel="cosine", sigma=None, weights=None, standardize=True,
                return_details=False, max_n=20000, nystrom_m=4000, random_state=0):
    """
    Vendi score for CyTOF (or any) data.
    - X: (n_cells, n_markers) numpy array.
    - weights: optional (n_cells,) nonnegative, will be normalized to sum 1.
    - If n is large, uses Nyström approximation with m landmarks.
    """
    n = X.shape[0]
    if standardize:
        X = StandardScaler(with_mean=True, with_std=True).fit_transform(X)

    # Nyström path for big n
    if n > max_n:
        rng = np.random.default_rng(random_state)
        m = min(nystrom_m, n)
        landmark_idx = rng.choice(n, m, replace=False)
        X_m = X[landmark_idx]
        K_mm = _kernel_matrix(X_m, kernel=kernel, sigma=sigma)
        # stabilize
        eps = 1e-8
        K_mm = (K_mm + K_mm.T) / 2.0 + eps * np.eye(m)
        # Cross-kernel to all points
        # Compute in batches to reduce memory if needed:
        batch = 20000
        Ks = []
        for a in range(0, n, batch):
            Ks.append(_kernel_matrix(X[a:a+batch], kernel=kernel, sigma=sigma)[:, :m]
                      if kernel == "cosine"
                      else rbf_kernel(X[a:a+batch], X_m, gamma=1.0/(2.0*(np.median(np.square(X_m[:,None,:]-X_m[None,:,:]).sum(-1))**0.5 if sigma is None else sigma)**2)))
        K_nm = np.vstack(Ks)  # (n, m)

        # Weighted Nyström spectrum:
        if weights is None:
            p = np.full(n, 1.0/n)
        else:
            p = np.asarray(weights).astype(float)
            p = p / p.sum()
        # Form A ≈ sqrt(P) * K_nm * K_mm^{-1} * K_mn * sqrt(P)
        P_sqrt = np.sqrt(p)[:, None]
        K_tilde = P_sqrt * K_nm  # (n,m)
        # solve K_mm^{-1} via eig or cho
        w, U = np.linalg.eigh(K_mm)
        w_inv = 1.0 / np.maximum(w, eps)
        Kmm_inv = (U * w_inv) @ U.T
        A_m = K_tilde.T @ K_tilde  # (m,m)  ~ K_mn P K_nm
        A = Kmm_inv @ A_m          # (m,m)  ~ K_mm^{-1} K_mn P K_nm
        # Eigenvalues of sqrt(P) K sqrt(P) are eigenvalues of A
        evals = np.clip(np.linalg.eigvalsh((A + A.T)/2.0), 0, None)
    else:
        K = _kernel_matrix(X, kernel=kernel, sigma=sigma)
        # Weighted variant: A = sqrt(P) K sqrt(P)
        if weights is None:
            A = K
        else:
            p = np.asarray(weights).astype(float)
            p = p / p.sum()
            A = (np.sqrt(p)[:, None]) * K * (np.sqrt(p)[None, :])
        A = (A + A.T) / 2.0
        evals = np.clip(np.linalg.eigvalsh(A), 0, None)

    # Normalize spectrum to sum to 1 (trace-normalization)
    s = evals.sum()
    if s <= 0:
        return 1.0 if not return_details else (1.0, {"lambdas": np.array([1.0])})
    lam = evals / s
    # Shannon entropy of spectrum; exp(entropy) = Vendi
    # Guard tiny zeros
    lam_nz = lam[lam > 0]
    vendi = float(np.exp(-np.sum(lam_nz * np.log(lam_nz))))
    if return_details:
        return vendi, {"lambdas": lam, "n": X.shape[0]}
    return vendi


def vendi_evenness(X, **kwargs):
    v = vendi_score(X, **kwargs)
    n = X.shape[0]
    J = float(np.log(v) / np.log(n)) if n > 1 else 0.0
    return v, J

def vendi_rarefied(X, m=5000, n_reps=200, random_state=0, return_all=False, **kwargs):
    rng = np.random.default_rng(random_state)
    n = X.shape[0]
    m = min(m, n)
    vals = []
    for _ in range(n_reps):
        idx = rng.choice(n, m, replace=False)
        MM=KBinsDiscretizer(strategy='uniform',encode='ordinal',n_bins=min(np.int32(m/2),20)).fit_transform(X[idx])
        V=vendi.score_dual(MM,normalize=True)
        vals.append(V)
    vals = np.array(vals, float)
    if return_all:
        return vals.mean(), np.quantile(vals, [0.025, 0.975]),vals
    else:
        return vals.mean(), np.quantile(vals, [0.025, 0.975])


In [ ]:
MRK=NamesAll.copy()
MRK.remove('H3')
MRK.remove('H3.3')
MRK.remove('H4')

In [ ]:
from icecream import ic
from sklearn.preprocessing import KBinsDiscretizer

In [ ]:
CLR={'Cycling':'#3f78c1','Basal-like':'#fb9a99','Luminal':'#33a02c'}

In [ ]:
from collections import Counter 
Counter(PDX8['Class'])

In [ ]:
for DB in tqdm(DBs[:]):
    ax=plt.subplot()
    CAll=globals()[f"{DB}"].copy()
    # UM=umap.UMAP(min_dist=0.001,n_neighbors=50,random_state=None,verbose=False)
    # X_2d=UM.fit_transform(CAll[NamesAll])
    AD=ad.AnnData(CAll[NamesAll],obs=CAll[['Samp','Class']])
    # AD.obsm['X_umap']=X_2d
    AD=AD[AD.obs['Class']!='Noise']
    m=np.int32(np.floor(AD.obs['Class'].value_counts().min()/2))
    print(m)
    for T in ['Luminal', 'Basal-like']:
        M=AD.obs['Class']==T
        ic(T)
        _,_,V=vendi_rarefied(AD[M,MRK].X,m=m,n_reps=5000,return_all=True, kernel="rbf", sigma=.1, 
                             random_state=None,standardize=True)
        #sns.histplot(np.log(V)/np.log(m),**hKWD,label=T,ax=ax,color=CLR[T])
        sns.histplot(V,**hKWD,label=T,ax=ax,color=CLR[T])
    plt.legend()
    plt.title(f"{DB}")
    plt.xlabel('Vendi Score')
    plt.show()

In [ ]:
MRK_CI=[
 'CD24',
 'CD44',
 'CD49f',
 'E-cadherin',
 'ER',

 'EpCAM',
 'GATA3',
 'KRT5',
 'KRT8-18',
 'Pan-KRT',
 'Vimentin',
 'aSMA',
]

In [ ]:
MRK_Epi=[
 'H2AK119ub',
 'H3K27ac',
 'H3K27me2',
 'H3K27me3',
 'H3K36me2',
 'H3K36me3',
 'H3K4me1',
 'H3K4me3',
 'H3K64ac',
 'H3K9ac',
 'H3K9me2',
 'H3K9me3',
 'H3S28p',
 'H4K16ac',
 'H4K20me3',

 'pH2A.X']

In [ ]:
import math
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

# --- settings ---
CLASSES = ['Luminal', 'Basal-like', 'Cycling']
NCOLS = 3                          # grid width; change if you like
HIST_KW = dict(element='step', stat='density', common_norm=False, bins=40, alpha=0.5)  # you can tweak
# HIST_KW can be replaced with your hKWD if you prefer: HIST_KW = {**hKWD}

# --- first pass: compute & cache V values per DB/class ---
cache = {}        # { DB: {class_label: np.array(V)} }
global_min = np.inf
global_max = -np.inf

for DB in tqdm(DBs[:], desc="Computing vendi distributions"):
    CAll = globals()[f"{DB}"].copy()
    AD = ad.AnnData(CAll[NamesAll], obs=CAll[['Samp','Class']])
    AD = AD[AD.obs['Class']!='Noise']

    # rarefaction size
    m = int(np.floor(AD.obs['Class'].value_counts().min()/2))
    print(f"{DB}: m={m}")

    cache[DB] = {}
    for T in CLASSES:
        M = (AD.obs['Class'] == T)
        if M.sum() < m or M.sum() == 0:
            # not enough cells to rarefy; skip plotting this class for this DB
            continue
        ic(T)
        _, _, V = vendi_rarefied(
            AD[M, MRK_Epi].X,
            m=m,
            n_reps=5000,
            return_all=True,
            kernel="rbf",
            sigma=.1,
            random_state=None,
            standardize=True
        )
        V = np.asarray(V).ravel()
        cache[DB][T] = V
        if len(V):
            global_min = min(global_min, V.min())
            global_max = max(global_max, V.max())

# If nothing was computed, bail out gracefully
if not cache or all(len(v)==0 for d in cache.values() for v in d.values()):
    raise RuntimeError("No valid Vendi distributions to plot (check class sizes vs m).")

# Define common bins and limits across all subplots
# (40 bins by default; adjust via HIST_KW['bins'])
if isinstance(HIST_KW.get('bins', 40), int):
    bins = np.linspace(global_min, global_max, HIST_KW.get('bins', 40))
else:
    bins = HIST_KW['bins']

# --- second pass: plot on a shared grid ---
n = len(DBs[:])
nrows = math.ceil(n / NCOLS)
fig, axes = plt.subplots(nrows, NCOLS, figsize=(5*NCOLS, 3.6*nrows), sharex=True, sharey=True)
axes = np.atleast_1d(axes).ravel()

# For a consistent legend, keep track of handles/labels once
legend_done = False
handles_labels = None

for i, DB in enumerate(DBs[:]):
    ax = axes[i]
    plotted_any = False
    for T in CLASSES:
        V = cache.get(DB, {}).get(T, None)
        if V is None or len(V) == 0:
            continue
        sns.histplot(V, ax=ax, bins=bins, label=T, color=CLR[T], fill=False,**{k:v for k,v in HIST_KW.items() if k!='bins'})
        plotted_any = True

    ax.set_title(f"{DB}")
    ax.set_xlabel('Vendi Score')
    ax.set_ylabel('Density')

    # Capture legend handles/labels from the first populated axis
    if plotted_any and not legend_done:
        handles, labels = ax.get_legend_handles_labels()
        handles_labels = (handles, labels)
        legend_done = True
    ax.legend_.remove() if ax.get_legend() else None

# Remove any empty axes if DB count doesn't fill the grid
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

# Add one global legend at the bottom (or top)
if handles_labels is not None:
    fig.legend(*handles_labels, loc='lower center', ncol=len(CLASSES), frameon=False, bbox_to_anchor=(0.5, -0.02))

fig.tight_layout()
#plt.savefig("Plots/Vendi_CI.png",dpi=200,bbox_inches='tight')
plt.show()


In [ ]:
BL=[]
BLh=[]
BLl=[]
BLs=[]
L=[]
Ll=[]
Lh=[]
Ls=[]
for k in cache.keys():
   BL.append(cache[k]['Basal-like'].mean())
   BLh.append(np.quantile(cache[k]['Basal-like'],0.975))
   BLl.append(np.quantile(cache[k]['Basal-like'],0.025))
   BLs.append(cache[k]['Basal-like'].std()) 
   L.append(cache[k]['Luminal'].mean())
   Lh.append(np.quantile(cache[k]['Luminal'],0.975))
   Ll.append(np.quantile(cache[k]['Luminal'],0.025))
   Ls.append(cache[k]['Luminal'].std())

In [ ]:
Q=['BL','BLh','BLl','L','Ll','Lh']
for q in Q:
    globals()[q]=np.asarray(globals()[q])

In [ ]:
Nkeys=len(cache.keys())

In [ ]:
Nkeys

In [ ]:
plt.errorbar(range(Nkeys),BL,yerr=[BL-BLl,BLh-BL],capsize=2,fmt='.',color=CLR['Basal-like'],label='Basal-like')
plt.errorbar(np.asarray(range(Nkeys))+0.2,L,yerr=[L-Ll,Lh-L],capsize=2,fmt='.',color=CLR['Luminal'],label='Luminal')
plt.xticks(range(Nkeys),labels=DBs,rotation=90)
#plt.savefig("Plots/Vendi_EpiMRK.pdf",dpi=200,bbox_inches='tight')
plt.legend()

In [ ]:
MRK

In [ ]:
import math
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

# Try to use SciPy's fast pdist; fall back to NumPy if not available
try:
    from scipy.spatial.distance import pdist
    _HAVE_SCIPY = True
except Exception:
    _HAVE_SCIPY = False

def _mean_pairwise_dist_numpy(Y: np.ndarray) -> float:
    """
    Mean pairwise Euclidean distance using NumPy (O(m^2) memory/time).
    Used only if SciPy is unavailable.
    """
    # (m, d) -> (m, m, d) diffs; for large m this is heavy, but OK as a fallback
    diff = Y[:, None, :] - Y[None, :, :]
    D = np.sqrt(np.sum(diff * diff, axis=-1))
    m = D.shape[0]
    if m < 2:
        return 0.0
    iu = np.triu_indices(m, k=1)
    return D[iu].mean()

def bootstrap_avg_pairdist(
    X: np.ndarray,
    m: int,
    n_reps: int = 1000,
    standardize: bool = True,
    random_state=None,
) -> np.ndarray:
    """
    Draw n_reps subsets of size m without replacement and return the
    mean pairwise Euclidean distance for each subset.
    """
    rng = np.random.default_rng(random_state)

    # Ensure dense float array
    if hasattr(X, "A"):  # sparse
        X = X.A
    X = np.asarray(X, dtype=float, order="C")

    if standardize:
        mu = X.mean(axis=0, keepdims=True)
        sd = X.std(axis=0, keepdims=True)
        sd[sd == 0] = 1.0
        X = (X - mu) / sd

    n = X.shape[0]
    if m > n:
        raise ValueError(f"m ({m}) cannot exceed number of rows ({n}).")

    out = np.empty(n_reps, dtype=float)
    for r in range(n_reps):
        idx = rng.choice(n, size=m, replace=False)
        Y = X[idx]
        if _HAVE_SCIPY:
            d = pdist(Y, metric="euclidean").mean() if Y.shape[0] > 1 else 0.0
        else:
            d = _mean_pairwise_dist_numpy(Y)
        out[r] = d
    return out

# --- settings ---
CLASSES = ['Luminal', 'Basal-like', 'Cycling']
NCOLS = 3
HIST_KW = dict(element='step', stat='density', common_norm=False, bins=40, alpha=0.5)

# --- first pass: compute & cache bootstrap distributions per DB/class ---
cache = {}        # { DB: {class_label: np.array(dist_means)} }
global_min = np.inf
global_max = -np.inf

for DB in tqdm(DBs[:], desc="Computing avg pairwise distances"):
    CAll = globals()[f"{DB}"].copy()
    AD = ad.AnnData(CAll[NamesAll], obs=CAll[['Samp','Class']])
    AD = AD[AD.obs['Class']!='Noise']

    # rarefaction size (same logic you had)
    m = int(np.floor(AD.obs['Class'].value_counts().min()/2))
    print(f"{DB}: m={m}")

    cache[DB] = {}
    for T in CLASSES:
        M = (AD.obs['Class'] == T)
        nT = int(M.sum())
        if nT < m or nT == 0:
            continue

        # Bootstrap average pairwise Euclidean distance
        dist_vals = bootstrap_avg_pairdist(
            AD[M, MRK].X,
            m=m,
            n_reps=10000,          # keep your original number; lower if too slow
            standardize=False,
            random_state=None
        )

        cache[DB][T] = dist_vals
        if dist_vals.size:
            global_min = min(global_min, float(dist_vals.min()))
            global_max = max(global_max, float(dist_vals.max()))

# If nothing was computed, bail out gracefully
if not cache or all(len(v)==0 for d in cache.values() for v in d.values()):
    raise RuntimeError("No valid distance distributions to plot (check class sizes vs m).")

# Define common bins/limits
if isinstance(HIST_KW.get('bins', 40), int):
    bins = np.linspace(global_min, global_max, HIST_KW.get('bins', 40))
else:
    bins = HIST_KW['bins']

# --- second pass: plot on a shared grid ---
n = len(DBs[:])
nrows = math.ceil(n / NCOLS)
fig, axes = plt.subplots(nrows, NCOLS, figsize=(5*NCOLS, 3.6*nrows), sharex=True, sharey=True)
axes = np.atleast_1d(axes).ravel()

legend_done = False
handles_labels = None

for i, DB in enumerate(DBs[:]):
    ax = axes[i]
    plotted_any = False
    for T in CLASSES:
        V = cache.get(DB, {}).get(T, None)
        if V is None or len(V) == 0:
            continue
        sns.histplot(
            V, ax=ax, bins=bins, label=T, color=CLR[T], fill=False,
            **{k:v for k,v in HIST_KW.items() if k!='bins'}
        )
        plotted_any = True

    ax.set_title(f"{DB}")
    ax.set_xlabel('Average pairwise distance (Euclidean)')
    ax.set_ylabel('Density')

    if plotted_any and not legend_done:
        handles, labels = ax.get_legend_handles_labels()
        handles_labels = (handles, labels)
        legend_done = True
    if ax.get_legend():
        ax.legend_.remove()

# Remove empty axes if grid not fully used
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

# Global legend
if handles_labels is not None:
    fig.legend(*handles_labels, loc='lower center', ncol=len(CLASSES),
               frameon=False, bbox_to_anchor=(0.5, -0.02))

fig.tight_layout()
# plt.savefig("Plots/AvgPairDist_CI.png", dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
AD

In [ ]:
M=summary.Class=='Basal-like'
y = summary[M]["mean"]
yerr_low = y.values - summary[M]["ci_low"].values
yerr_high = summary[M]["ci_high"].values - y.values
plt.errorbar(range(summary[M].shape[0]),y,yerr=[yerr_low, yerr_high],color=CLR['Basal-like'],fmt='.',capsize=2)
M=summary.Class=='Luminal'
y = summary[M]["mean"]
yerr_low = y.values - summary[M]["ci_low"].values
yerr_high = summary[M]["ci_high"].values - y.values
plt.errorbar(np.asarray(range(summary[M].shape[0]))+0.1,y,yerr=[yerr_low, yerr_high],color=CLR['Luminal'],fmt='.',capsize=2)



In [ ]:
M1=summary.Class=='Basal-like'
M2=summary.Class=='Luminal'


In [ ]:
import scipy

In [ ]:
scipy.stats.mannwhitneyu(summary[M1]['mean'],summary[M2]['mean'])